# MatForge — Comparative Benchmark
**Quantitative evaluation of Pix2Pix, DeepPBR, MatForge (PBR prediction) and MatForge_SR (super-resolution)**

Evaluation protocol:
- PBR models are evaluated on the fixed MatForge validation split (SEED=42, ~487 textures)
- Pix2Pix and DeepPBR receive 256×256 center crops; MatForge receives full 1K images via tile-and-merge
- SR models are evaluated on 100 textures degraded ×4 bicubic, compared against HR ground truth
- Metrics: MAE angular (normal), MAE + RMSE (roughness, metallic), LPIPS on 5-light render, PSNR + SSIM + LPIPS (SR)
- All results exported as CSV

**Execution order:** TensorFlow models first → explicit GPU release → PyTorch models

---
## 0. Configuration — Edit paths here before running

In [1]:
# =============================================================================
# CONFIGURATION BLOCK — adjust all paths before running
# =============================================================================

from pathlib import Path

# --- Dataset root (MatSynth processed, 1024×1024) ---
DATASET_ROOT = Path('//kaggle/input/datasets/mjgut05/matforge-pbr-dataset/matforge_dataset')   # adjust to your dataset name
RGB_DIR      = DATASET_ROOT / 'maps' / 'rgb'
NORMAL_DIR   = DATASET_ROOT / 'maps' / 'normal'
ROUGH_DIR    = DATASET_ROOT / 'maps' / 'roughness'
METAL_DIR    = DATASET_ROOT / 'maps' / 'metallic'

# Path to the persisted train/val split CSV produced during MatForge training
SPLIT_CSV = Path('/kaggle/input/datasets/mjgut05/matforge-pbr-dataset/matforge_split.csv')

# --- Pix2Pix checkpoint ---
PIX2PIX_CKPT_DIR  = Path('/kaggle/input/notebooks/mjgut05/pix2pix/entrenamiento_pbr')
PIX2PIX_CKPT_NAME = 'ckpt-30'   # best checkpoint

# --- DeepPBR checkpoint ---
DEEPPBR_CKPT_DIR  = Path('/kaggle/input/notebooks/mjgut05/deeppbr-net/entrenamiento_pbr')
DEEPPBR_CKPT_NAME = 'ckpt-42'   # epoch 157, best checkpoint

# --- MatForge checkpoint (PyTorch) ---
MATFORGE_CKPT = Path('/kaggle/input/datasets/mjgut05/matforge-checkpoints-ep20/checkpoints/best_gan.pt')

# --- MatForge SR checkpoints (PyTorch) ---
SR_FT_CKPT   = Path('/kaggle/input/notebooks/mjgut05/matforge-sr-01-training/sr_checkpoints/sr_ft_phase1_best_lpips.pt')   # fine-tuned
SR_BASE_CKPT = Path('/kaggle/input/datasets/mjgut05/matforge-sr-weights/RealESRGAN_x4plus.pth')         # base weights

# --- Output directory ---
OUT_DIR = Path('/kaggle/working/benchmark_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Evaluation constants ---
SEED          = 42
SR_SAMPLE_N   = 100    # number of textures for SR evaluation
IMG_SIZE_TF   = 256    # fixed input size for Pix2Pix and DeepPBR
IMG_SIZE_FULL = 1024   # full resolution for MatForge
N_LIGHTS      = 5      # number of fixed lights for render-based LPIPS

print('Configuration loaded.')
print(f'  Dataset root : {DATASET_ROOT}')
print(f'  Output dir   : {OUT_DIR}')

Configuration loaded.
  Dataset root : //kaggle/input/datasets/mjgut05/matforge-pbr-dataset/matforge_dataset
  Output dir   : /kaggle/working/benchmark_results


---
## 1. Shared utilities — metrics, rendering, data loading

In [2]:
# Shared metric and rendering utilities used by all evaluation sections.
# Kept in a single cell so they are available regardless of execution order.

# Install dependencies not available by default in the Kaggle environment
import subprocess
subprocess.run(['pip', 'install', 'lpips', '--quiet'], check=True)

import numpy as np
import pandas as pd
from PIL import Image
import lpips
import torch
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

# LPIPS metric (AlexNet backbone, as specified in the evaluation protocol)
_lpips_fn = lpips.LPIPS(net='alex')


def angular_mae(pred_norm: np.ndarray, gt_norm: np.ndarray) -> float:
    """Mean angular error in degrees between two normal maps.

    Both inputs are expected in [-1, 1] range with shape (H, W, 3).
    L2 normalisation is applied before the dot product to guard against
    near-zero vectors that would otherwise produce NaN in arccos.
    """
    pred = pred_norm / (np.linalg.norm(pred_norm, axis=-1, keepdims=True) + 1e-8)
    gt   = gt_norm   / (np.linalg.norm(gt_norm,   axis=-1, keepdims=True) + 1e-8)
    dot  = np.clip((pred * gt).sum(axis=-1), -1.0, 1.0)
    return float(np.degrees(np.arccos(dot)).mean())


def scalar_mae(pred: np.ndarray, gt: np.ndarray) -> float:
    """Mean absolute error for scalar maps in [0, 1]."""
    return float(np.abs(pred - gt).mean())


def scalar_rmse(pred: np.ndarray, gt: np.ndarray) -> float:
    """Root mean squared error for scalar maps in [0, 1]."""
    return float(np.sqrt(((pred - gt) ** 2).mean()))


def compute_lpips(img_a: np.ndarray, img_b: np.ndarray) -> float:
    """LPIPS distance between two RGB images in [0, 1] with shape (H, W, 3).

    Converts to [-1, 1] tensors as expected by the lpips library.
    """
    def to_tensor(img):
        t = torch.from_numpy(img).permute(2, 0, 1).float() * 2.0 - 1.0
        return t.unsqueeze(0)
    with torch.no_grad():
        return float(_lpips_fn(to_tensor(img_a), to_tensor(img_b)).item())


# Fixed light directions for render-based LPIPS (5 lights, elevation + azimuth pairs)
# Directions are unit vectors in world space: (x, y, z) with y-up convention.
_LIGHT_DIRS = np.array([
    [ 0.0,  0.0,  1.0],   # frontal
    [ 0.577,  0.577, 0.577],
    [-0.577,  0.577, 0.577],
    [ 0.577, -0.577, 0.577],
    [-0.577, -0.577, 0.577],
], dtype=np.float32)


def _cook_torrance_render(
    albedo: np.ndarray,
    normal: np.ndarray,
    roughness: np.ndarray,
    metallic: np.ndarray,
    light_dir: np.ndarray,
) -> np.ndarray:
    """Simplified Cook-Torrance render for a single directional light.

    Uses the GGX normal distribution function and Schlick approximations.
    All inputs in [0, 1] except normal which is in [-1, 1] (OpenGL convention).
    Returns an RGB render in [0, 1].

    This implementation follows the PBR formulation used in the MatForge
    training render loss, ensuring consistency between training and evaluation.
    """
    # Normalise inputs
    N = normal / (np.linalg.norm(normal, axis=-1, keepdims=True) + 1e-8)
    L = light_dir / (np.linalg.norm(light_dir) + 1e-8)
    V = np.array([0.0, 0.0, 1.0], dtype=np.float32)  # camera at z+
    H = (L + V) / (np.linalg.norm(L + V) + 1e-8)

    NdotL = np.clip((N * L).sum(axis=-1, keepdims=True), 0.0, 1.0)
    NdotV = np.clip((N * V).sum(axis=-1, keepdims=True), 1e-4, 1.0)
    NdotH = np.clip((N * H).sum(axis=-1, keepdims=True), 0.0, 1.0)
    VdotH = np.clip((V * H).sum(keepdims=True),          0.0, 1.0)

    alpha  = roughness ** 2
    alpha2 = alpha ** 2

    # GGX normal distribution
    denom = NdotH ** 2 * (alpha2 - 1.0) + 1.0
    D = alpha2 / (np.pi * denom ** 2 + 1e-8)

    # Schlick-GGX geometry
    k  = (roughness + 1.0) ** 2 / 8.0
    G1 = NdotV / (NdotV * (1.0 - k) + k + 1e-8)
    G2 = NdotL / (NdotL * (1.0 - k) + k + 1e-8)
    G  = G1 * G2

    # Schlick Fresnel
    F0 = 0.04 * (1.0 - metallic) + albedo * metallic
    F  = F0 + (1.0 - F0) * (1.0 - VdotH) ** 5

    specular = (D * G * F) / (4.0 * NdotV * NdotL + 1e-8)
    diffuse  = albedo / np.pi * (1.0 - F) * (1.0 - metallic)

    render = (diffuse + specular) * NdotL
    return np.clip(render, 0.0, 1.0)


def render_lpips(
    albedo_gt: np.ndarray,
    normal_pred: np.ndarray,
    roughness_pred: np.ndarray,
    metallic_pred: np.ndarray,
    normal_gt: np.ndarray,
    roughness_gt: np.ndarray,
    metallic_gt: np.ndarray,
) -> float:
    """LPIPS between renders of predicted and GT maps under 5 fixed lights.

    Albedo GT is shared between predicted and GT renders so that differences
    in the render reflect exclusively Normal, Roughness and Metallic prediction
    quality, as specified in the evaluation protocol (Doc3, section 15).
    """
    lpips_values = []
    for light in _LIGHT_DIRS:
        render_pred = _cook_torrance_render(
            albedo_gt, normal_pred, roughness_pred, metallic_pred, light
        )
        render_gt = _cook_torrance_render(
            albedo_gt, normal_gt, roughness_gt, metallic_gt, light
        )
        lpips_values.append(compute_lpips(render_pred, render_gt))
    return float(np.mean(lpips_values))


print('Shared utilities loaded.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.7 MB/s eta 0:00:00
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 206MB/s]


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
Shared utilities loaded.


---
## 2. Dataset — load validation split

In [3]:
# Loads the fixed train/val split CSV produced during MatForge training.
# All models are evaluated on exactly the same set of textures to ensure
# a fair comparison.

import os

split_df = pd.read_csv(SPLIT_CSV)
val_df   = split_df[split_df['split'] == 'val'].reset_index(drop=True)

print(f'Validation textures : {len(val_df)}')
print(f'Group distribution  :')
print(val_df['functional_group'].value_counts().to_string())

# SR subset: 100 textures sampled with SEED=42, stratified by group
sr_df = val_df.groupby('functional_group', group_keys=False).apply(
    lambda g: g.sample(frac=SR_SAMPLE_N / len(val_df), random_state=SEED)
).reset_index(drop=True)
# Ensure exactly SR_SAMPLE_N textures (rounding may give ±1)
sr_df = sr_df.sample(n=min(SR_SAMPLE_N, len(sr_df)), random_state=SEED).reset_index(drop=True)

print(f'\nSR evaluation subset: {len(sr_df)} textures')

Validation textures : 483
Group distribution  :
functional_group
mixed_ambiguous     116
wood                 98
ceramic_ground       75
stone_rough          71
brick_terracotta     41
metal                35
marble_smooth        28
concrete_plaster     19

SR evaluation subset: 100 textures


/tmp/ipykernel_23/212447120.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sr_df = val_df.groupby('functional_group', group_keys=False).apply(


---
## 3. TensorFlow models — Pix2Pix and DeepPBR

> Run sections 3.1–3.3 before loading any PyTorch model.
> GPU memory is released explicitly at the end of section 3.3.

### 3.1 Pix2Pix — model definition and checkpoint restore

In [4]:
import tensorflow as tf

IMG_HEIGHT = IMG_SIZE_TF
IMG_WIDTH  = IMG_SIZE_TF
OUTPUT_CHANNELS = 4
initializer = tf.random_normal_initializer(0.0, 0.02)


def upsample(filters, size, apply_dropout=False):
    result = tf.keras.Sequential()
    result.add(tf.keras.layers.Conv2DTranspose(
        filters, size, strides=2, padding='same',
        kernel_initializer=initializer, use_bias=False
    ))
    result.add(tf.keras.layers.BatchNormalization())
    if apply_dropout:
        result.add(tf.keras.layers.Dropout(0.5))
    result.add(tf.keras.layers.ReLU())
    return result


def downsample(filters, size, apply_batchnorm=True):
    result = tf.keras.Sequential()
    result.add(tf.keras.layers.Conv2D(
        filters, size, strides=2, padding='same',
        kernel_initializer=initializer, use_bias=False
    ))
    if apply_batchnorm:
        result.add(tf.keras.layers.BatchNormalization())
    result.add(tf.keras.layers.LeakyReLU())
    return result


def Discriminator():
    inp   = tf.keras.layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3], name='input_image')
    tar   = tf.keras.layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 4], name='target_image')
    x     = tf.keras.layers.concatenate([inp, tar])
    down1 = downsample(64,  4, apply_batchnorm=False)(x)
    down2 = downsample(128, 4)(down1)
    down3 = downsample(256, 4)(down2)
    zero_pad1  = tf.keras.layers.ZeroPadding2D()(down3)
    conv       = tf.keras.layers.Conv2D(512, 4, strides=1,
                     kernel_initializer=initializer, use_bias=False)(zero_pad1)
    batchnorm1 = tf.keras.layers.BatchNormalization()(conv)
    leaky_relu = tf.keras.layers.LeakyReLU()(batchnorm1)
    zero_pad2  = tf.keras.layers.ZeroPadding2D()(leaky_relu)
    last       = tf.keras.layers.Conv2D(1, 4, strides=1,
                     kernel_initializer=initializer)(zero_pad2)
    return tf.keras.Model(inputs=[inp, tar], outputs=last, name='Discriminador_PatchGAN')


def Generator():
    inputs     = tf.keras.layers.Input(shape=[IMG_HEIGHT, IMG_WIDTH, 3], name='input_image')
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=[IMG_HEIGHT, IMG_WIDTH, 3], include_top=False, weights='imagenet'
    )
    layer_names = [
        'block_1_expand_relu',
        'block_3_expand_relu',
        'block_6_expand_relu',
        'block_13_expand_relu',
        'block_16_project',
    ]
    base_model_outputs = [base_model.get_layer(n).output for n in layer_names]
    down_stack = tf.keras.Model(
        inputs=base_model.input, outputs=base_model_outputs, name='encoder_mobilenet'
    )
    down_stack.trainable = False
    up_stack = [
        upsample(512, 3, apply_dropout=True),
        upsample(256, 3, apply_dropout=True),
        upsample(128, 3),
        upsample(64,  3),
    ]
    x     = inputs
    skips = down_stack(x)
    x     = skips[-1]
    skips = reversed(skips[:-1])
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = tf.keras.layers.Concatenate()([x, skip])
    last = tf.keras.layers.Conv2DTranspose(
        filters=OUTPUT_CHANNELS, kernel_size=3, strides=2, padding='same',
        kernel_initializer=initializer, activation='tanh'
    )
    x = last(x)
    return tf.keras.Model(inputs=inputs, outputs=x, name='Generador_PBR')


pix2pix_gen  = Generator()
pix2pix_disc = Discriminator()
gen_opt_p2p  = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
disc_opt_p2p = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

ckpt_p2p = tf.train.Checkpoint(
    generator_optimizer=gen_opt_p2p,
    discriminator_optimizer=disc_opt_p2p,
    generador=pix2pix_gen,
    discriminador=pix2pix_disc,
)
ckpt_p2p.restore(
    str(PIX2PIX_CKPT_DIR / PIX2PIX_CKPT_NAME)
).expect_partial()

print(f'Pix2Pix checkpoint restored: {PIX2PIX_CKPT_NAME}')

2026-05-15 00:27:08.416632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778804828.624620      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778804828.687240      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778804829.203570      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778804829.203604      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778804829.203607      23 computation_placer.cc:177] computation placer alr

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Pix2Pix checkpoint restored: ckpt-30


### 3.2 DeepPBR — model definition and checkpoint restore

In [5]:
from tensorflow.keras import layers
import tensorflow as tf


def cbam_block(input_feature, ratio=8):
    channels         = input_feature.shape[-1]
    shared_layer_one = layers.Dense(channels // ratio, activation='relu',
                                    kernel_initializer='he_normal', use_bias=True)
    shared_layer_two = layers.Dense(channels, kernel_initializer='he_normal', use_bias=True)
    avg_pool         = layers.GlobalAveragePooling2D()(input_feature)
    avg_pool         = layers.Reshape((1, 1, channels))(avg_pool)
    avg_pool         = shared_layer_two(shared_layer_one(avg_pool))
    max_pool         = layers.GlobalMaxPooling2D()(input_feature)
    max_pool         = layers.Reshape((1, 1, channels))(max_pool)
    max_pool         = shared_layer_two(shared_layer_one(max_pool))
    cbam_feature     = layers.Multiply()(
        [input_feature,
         layers.Activation('sigmoid')(layers.Add()([avg_pool, max_pool]))]
    )
    spatial_avg  = layers.Lambda(
        lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(cbam_feature)
    spatial_max  = layers.Lambda(
        lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(cbam_feature)
    concat       = layers.Concatenate(axis=-1)([spatial_avg, spatial_max])
    cbam_feature = layers.Multiply()(
        [cbam_feature,
         layers.Activation('sigmoid')(
             layers.Conv2D(1, (7, 7), padding='same', activation=None)(concat)
         )]
    )
    return cbam_feature


def upsample_block(filters, size, apply_dropout=False, apply_attention=True):
    initializer = tf.random_normal_initializer(0., 0.02)
    result      = tf.keras.Sequential()
    result.add(layers.Conv2DTranspose(filters, size, strides=2, padding='same',
                                      kernel_initializer=initializer, use_bias=False))
    result.add(layers.BatchNormalization())
    if apply_dropout:
        result.add(layers.Dropout(0.5))
    result.add(layers.ReLU())
    return result


def build_deep_pbr_generator():
    inputs     = layers.Input(shape=[256, 256, 3])
    base_model = tf.keras.applications.ResNet50(
        input_tensor=inputs, include_top=False, weights='imagenet'
    )
    base_model.trainable = True
    for layer in base_model.layers[:80]:
        layer.trainable = False
    skip_names = ['conv1_relu', 'conv2_block3_out', 'conv3_block4_out', 'conv4_block6_out']
    skips      = [base_model.get_layer(name).output for name in skip_names]
    bottleneck = base_model.get_layer('conv5_block3_out').output

    def build_decoder_head(x, head_name, output_channels):
        up_filters  = [1024, 512, 256, 64]
        initializer = tf.random_normal_initializer(0., 0.02)
        for skip, filters in zip(reversed(skips), up_filters):
            x             = upsample_block(filters, 4)(x)
            skip_attended = cbam_block(skip)
            x             = layers.Concatenate()([x, skip_attended])
        last = layers.Conv2DTranspose(
            output_channels, 4, strides=2, padding='same',
            kernel_initializer=initializer, activation='tanh',
            name=head_name, dtype='float32'
        )
        return last(x)

    normal_output    = build_decoder_head(bottleneck, 'salida_normal',    3)
    roughness_output = build_decoder_head(bottleneck, 'salida_roughness', 1)
    return tf.keras.Model(
        inputs=inputs, outputs=[normal_output, roughness_output],
        name='DeepPBR_Generator'
    )


def build_discriminator():
    init        = tf.random_normal_initializer(0.0, 0.02)
    inp         = layers.Input(shape=[256, 256, 3], name='input_rgb')
    tar_normal  = layers.Input(shape=[256, 256, 3], name='target_normal')
    tar_rough   = layers.Input(shape=[256, 256, 1], name='target_roughness')
    x           = layers.Concatenate()([inp, tar_normal, tar_rough])
    down1       = downsample_block(64,  4, apply_batchnorm=False)(x)
    down2       = downsample_block(128, 4)(down1)
    down3       = downsample_block(256, 4)(down2)
    zero_pad1   = layers.ZeroPadding2D()(down3)
    conv        = layers.Conv2D(512, 4, strides=1,
                      kernel_initializer=init, use_bias=False)(zero_pad1)
    batchnorm1  = layers.BatchNormalization()(conv)
    leaky_relu  = layers.LeakyReLU()(batchnorm1)
    zero_pad2   = layers.ZeroPadding2D()(leaky_relu)
    last        = layers.Conv2D(1, 4, strides=1,
                      kernel_initializer=init, dtype='float32')(zero_pad2)
    return tf.keras.Model(
        inputs=[inp, tar_normal, tar_rough], outputs=last,
        name='Discriminador_PatchGAN'
    )


def downsample_block(filters, size, apply_batchnorm=True):
    init   = tf.random_normal_initializer(0.0, 0.02)
    result = tf.keras.Sequential()
    result.add(layers.Conv2D(filters, size, strides=2, padding='same',
                             kernel_initializer=init, use_bias=False))
    if apply_batchnorm:
        result.add(layers.BatchNormalization())
    result.add(layers.LeakyReLU())
    return result


deeppbr_gen  = build_deep_pbr_generator()
deeppbr_disc = build_discriminator()
gen_opt_dpbr  = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
disc_opt_dpbr = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

ckpt_dpbr = tf.train.Checkpoint(
    generator_optimizer=gen_opt_dpbr,
    discriminator_optimizer=disc_opt_dpbr,
    generador_pbr=deeppbr_gen,
    discriminador_pbr=deeppbr_disc,
)
ckpt_dpbr.restore(
    str(DEEPPBR_CKPT_DIR / DEEPPBR_CKPT_NAME)
).expect_partial()

print(f'DeepPBR checkpoint restored: {DEEPPBR_CKPT_NAME}')
print(f'Model weights: {len(deeppbr_gen.weights)}')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
DeepPBR checkpoint restored: ckpt-42
Model weights: 410


### 3.3 TF inference loop — Pix2Pix and DeepPBR

In [6]:
# Runs inference for both TF models over the validation split.
# Each texture is centre-cropped to 256×256 before inference, matching
# the training input conditions of both models.
# Predictions are stored as float32 numpy arrays in [0, 1] on disk
# so that the PyTorch section can load them without keeping TF in memory.

import gc


def centre_crop_256(img_np: np.ndarray) -> np.ndarray:
    """Extract a 256×256 centre crop from a (H, W, C) array."""
    h, w = img_np.shape[:2]
    top  = (h - IMG_SIZE_TF) // 2
    left = (w - IMG_SIZE_TF) // 2
    return img_np[top:top + IMG_SIZE_TF, left:left + IMG_SIZE_TF]


def load_image_np(path: Path) -> np.ndarray:
    """Load a PNG as a float32 array in [0, 1] with shape (H, W, C)."""
    return np.array(Image.open(path).convert('RGB')).astype(np.float32) / 255.0


def load_normal_np(path: Path) -> np.ndarray:
    """Load a normal map PNG and convert to [-1, 1] range."""
    arr = np.array(Image.open(path).convert('RGB')).astype(np.float32) / 255.0
    return arr * 2.0 - 1.0


def load_scalar_np(path: Path) -> np.ndarray:
    """Load a single-channel map (roughness / metallic) as float32 in [0, 1]."""
    img = Image.open(path).convert('L')
    return np.array(img).astype(np.float32)[:, :, None] / 255.0


rows_p2p  = []
rows_dpbr = []

for idx, row in val_df.iterrows():
    stem = row['filename']   # e.g. 'stone_rough_0001'

    # --- Load ground-truth maps ---
    rgb_full   = load_image_np(RGB_DIR    / stem)
    normal_gt  = load_normal_np(NORMAL_DIR / stem)    # [-1, 1]
    rough_gt   = load_scalar_np(ROUGH_DIR  / stem)    # [0, 1]

    # Metallic GT: zero for non-metal groups; real map for 'metal' group
    metal_path = METAL_DIR / stem
    if metal_path.exists():
        metal_gt = load_scalar_np(metal_path)
    else:
        metal_gt = np.zeros_like(rough_gt)

    # --- Centre crop all maps to 256×256 ---
    rgb_crop    = centre_crop_256(rgb_full)
    normal_crop = centre_crop_256(normal_gt)
    rough_crop  = centre_crop_256(rough_gt)
    metal_crop  = centre_crop_256(metal_gt)
    albedo_crop = rgb_crop   # albedo GT = RGB input (as per evaluation protocol)

    # Convert RGB to [-1, 1] for TF models
    tf_input = rgb_crop * 2.0 - 1.0
    tf_input_t = tf.expand_dims(tf.constant(tf_input, dtype=tf.float32), 0)

    # --- Pix2Pix inference ---
    p2p_out   = pix2pix_gen(tf_input_t, training=False)[0].numpy()   # (256,256,4)
    p2p_norm  = (p2p_out[:, :, :3] + 1.0) / 2.0 * 2.0 - 1.0        # keep in [-1,1]
    p2p_rough = (p2p_out[:, :, 3:4] + 1.0) / 2.0                    # to [0,1]

    rows_p2p.append({
        'filename'   : stem,
        'group'      : row['functional_group'],
        'normal_mae' : angular_mae(p2p_norm, normal_crop),
        'rough_mae'  : scalar_mae(p2p_rough, rough_crop),
        'rough_rmse' : scalar_rmse(p2p_rough, rough_crop),
        'lpips_render': render_lpips(
            albedo_crop,
            p2p_norm,   p2p_rough,  np.zeros_like(rough_crop),
            normal_crop, rough_crop, metal_crop
        ),
    })

    # --- DeepPBR inference ---
    dpbr_norm_t, dpbr_rough_t = deeppbr_gen(tf_input_t, training=False)
    dpbr_norm  = dpbr_norm_t[0].numpy()                              # [-1,1]
    dpbr_rough = (dpbr_rough_t[0].numpy() + 1.0) / 2.0              # to [0,1]

    rows_dpbr.append({
        'filename'   : stem,
        'group'      : row['functional_group'],
        'normal_mae' : angular_mae(dpbr_norm, normal_crop),
        'rough_mae'  : scalar_mae(dpbr_rough, rough_crop),
        'rough_rmse' : scalar_rmse(dpbr_rough, rough_crop),
        'lpips_render': render_lpips(
            albedo_crop,
            dpbr_norm,  dpbr_rough, np.zeros_like(rough_crop),
            normal_crop, rough_crop, metal_crop
        ),
    })

    if idx % 50 == 0:
        print(f'  [{idx+1}/{len(val_df)}] processed')

df_p2p  = pd.DataFrame(rows_p2p)
df_dpbr = pd.DataFrame(rows_dpbr)

df_p2p.to_csv(OUT_DIR / 'results_pix2pix.csv',  index=False)
df_dpbr.to_csv(OUT_DIR / 'results_deeppbr.csv', index=False)

print('TF inference complete. Releasing GPU memory.')

# Explicit GPU release before loading PyTorch models
del pix2pix_gen, pix2pix_disc, deeppbr_gen, deeppbr_disc
del ckpt_p2p, ckpt_dpbr
del gen_opt_p2p, disc_opt_p2p, gen_opt_dpbr, disc_opt_dpbr
gc.collect()
tf.keras.backend.clear_session()
print('TF GPU memory released.')

I0000 00:00:1778804863.735102      23 cuda_dnn.cc:529] Loaded cuDNN version 91002


  [1/483] processed
  [51/483] processed
  [101/483] processed
  [151/483] processed
  [201/483] processed
  [251/483] processed
  [301/483] processed
  [351/483] processed
  [401/483] processed
  [451/483] processed
TF inference complete. Releasing GPU memory.
TF GPU memory released.


---
## 4. PyTorch models — MatForge and MatForge_SR

### 4.1 MatForge — model definition and checkpoint load

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float32
print(f'PyTorch device: {DEVICE}')


class FPNDecoder(nn.Module):
    def __init__(self, in_channels=(64, 128, 320, 512), out_channels=256):
        super().__init__()
        self.proj = nn.ModuleList([
            nn.Conv2d(c, out_channels, 1, bias=False) for c in in_channels
        ])
        self.merge = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(out_channels * 2, out_channels, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
            for _ in range(len(in_channels) - 1)
        ])

    def forward(self, features):
        p = [self.proj[i](f) for i, f in enumerate(features)]
        x = p[3]
        for i in range(2, -1, -1):
            x = F.interpolate(x, size=p[i].shape[-2:], mode='bilinear', align_corners=False)
            x = self.merge[i](torch.cat([x, p[i]], dim=1))
        return x


class RefineHead(nn.Module):
    def __init__(self, in_channels=256, out_channels=3):
        super().__init__()

        def up_block(ic, mid=128, oc=64):
            return nn.Sequential(
                nn.Conv2d(ic,  mid, 3, padding=1, bias=False),
                nn.BatchNorm2d(mid), nn.ReLU(inplace=True),
                nn.Conv2d(mid, oc,  3, padding=1, bias=False),
                nn.BatchNorm2d(oc),  nn.ReLU(inplace=True),
            )

        self.block1 = up_block(in_channels)
        self.block2 = up_block(64)
        self.out    = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        x = self.block1(F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False))
        x = self.block2(F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False))
        return self.out(x)


class MatForgeNet(nn.Module):
    def __init__(self, fpn_channels: int = 256):
        super().__init__()
        self.encoder = timm.create_model(
            'pvt_v2_b1', pretrained=False, features_only=True
        )
        enc_channels        = self.encoder.feature_info.channels()
        self.fpn            = FPNDecoder(enc_channels, fpn_channels)
        self.head_normal    = RefineHead(fpn_channels, 3)
        self.head_roughness = RefineHead(fpn_channels, 1)
        self.head_metallic  = RefineHead(fpn_channels, 1)

    def forward(self, x: torch.Tensor) -> dict:
        features  = self.encoder(x)
        fpn_feat  = self.fpn(features)
        normal    = F.normalize(torch.tanh(self.head_normal(fpn_feat)), dim=1, eps=1e-6)
        roughness = torch.sigmoid(self.head_roughness(fpn_feat))
        metallic  = self.head_metallic(fpn_feat)
        return {'normal': normal, 'roughness': roughness, 'metallic': metallic}


# --- Load checkpoint ---
ckpt_mf        = torch.load(MATFORGE_CKPT, map_location='cpu')
matforge_model = MatForgeNet().to(DEVICE).to(DTYPE)

# Load base model weights first (contains BatchNorm running stats)
# then override with EMA weights where available — EMA shadow omits
# running_mean/running_var which are not learnable parameters.
base_state = ckpt_mf['model']
matforge_model.load_state_dict(base_state, strict=False)

if 'ema_shadow' in ckpt_mf and ckpt_mf['ema_shadow']:
    ema_state = ckpt_mf['ema_shadow']
    current   = matforge_model.state_dict()
    current.update({k: v for k, v in ema_state.items() if k in current})
    matforge_model.load_state_dict(current)
    print('MatForge: base weights + EMA override loaded.')
else:
    print('MatForge: base weights loaded.')

matforge_model.eval()
print('MatForge ready.')

PyTorch device: cuda
MatForge: base weights + EMA override loaded.
MatForge ready.


### 4.2 MatForge — Hann tile-and-merge inference

In [8]:
# Tile-and-merge with Hann window blending, matching the inference pipeline
# described in the architecture document (tile 256×256, stride 128, symmetric
# TILE//2 padding on all borders).

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=DTYPE).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=DTYPE).view(3, 1, 1)
TILE   = 256
STRIDE = 128


def _hann2d(size: int) -> torch.Tensor:
    """2D Hann window of shape (size, size)."""
    w = torch.hann_window(size, periodic=False)
    return w.unsqueeze(0) * w.unsqueeze(1)


_HANN = _hann2d(TILE).to(DEVICE)   # cached


@torch.no_grad()
def matforge_infer(rgb_np: np.ndarray) -> dict:
    """Full tile-and-merge inference for an arbitrary-resolution RGB image.

    Input:  float32 numpy array in [0, 1], shape (H, W, 3)
    Output: dict with keys 'normal' ([-1,1]), 'roughness' ([0,1]), 'metallic' ([0,1])
            all as float32 numpy arrays of shape (H, W, C)
    """
    h, w = rgb_np.shape[:2]
    pad  = TILE // 2

    # Convert to tensor and apply ImageNet normalisation
    t = torch.from_numpy(rgb_np).permute(2, 0, 1).to(DEVICE).to(DTYPE)
    t = (t - IMAGENET_MEAN.to(DEVICE)) / IMAGENET_STD.to(DEVICE)

    # Symmetric padding
    t_pad = F.pad(t.unsqueeze(0), (pad, pad, pad, pad), mode='reflect').squeeze(0)
    _, hp, wp = t_pad.shape

    # Accumulation buffers
    acc_normal   = torch.zeros(3, hp, wp, device=DEVICE, dtype=DTYPE)
    acc_roughness = torch.zeros(1, hp, wp, device=DEVICE, dtype=DTYPE)
    acc_metallic  = torch.zeros(1, hp, wp, device=DEVICE, dtype=DTYPE)
    acc_weight    = torch.zeros(1, hp, wp, device=DEVICE, dtype=DTYPE)

    ys = list(range(0, hp - TILE + 1, STRIDE))
    xs = list(range(0, wp - TILE + 1, STRIDE))
    if ys[-1] + TILE < hp:
        ys.append(hp - TILE)
    if xs[-1] + TILE < wp:
        xs.append(wp - TILE)

    for y in ys:
        for x in xs:
            tile  = t_pad[:, y:y+TILE, x:x+TILE].unsqueeze(0)
            out   = matforge_model(tile)
            w_hat = _HANN.unsqueeze(0)
            acc_normal[   :, y:y+TILE, x:x+TILE] += out['normal'][0]    * w_hat
            acc_roughness[:, y:y+TILE, x:x+TILE] += out['roughness'][0] * w_hat
            acc_metallic[ :, y:y+TILE, x:x+TILE] += out['metallic'][0]  * w_hat
            acc_weight[   :, y:y+TILE, x:x+TILE] += w_hat

    eps = 1e-8
    merged_normal    = acc_normal    / (acc_weight + eps)
    merged_roughness = acc_roughness / (acc_weight + eps)
    merged_metallic  = acc_metallic  / (acc_weight + eps)

    # Renormalise normal map after Hann merge (correct order: merge first, then L2)
    merged_normal = F.normalize(merged_normal, dim=0, eps=1e-6)

    # Remove padding
    merged_normal    = merged_normal[   :, pad:pad+h, pad:pad+w]
    merged_roughness = merged_roughness[:, pad:pad+h, pad:pad+w]
    merged_metallic  = merged_metallic[ :, pad:pad+h, pad:pad+w]

    # Sigmoid on metallic logits; clip all to valid ranges
    merged_metallic  = torch.sigmoid(merged_metallic)
    merged_roughness = merged_roughness.clamp(0.0, 1.0)
    merged_metallic  = merged_metallic.clamp(0.0, 1.0)

    def to_np_hwc(t): return t.cpu().float().numpy().transpose(1, 2, 0)

    return {
        'normal'   : to_np_hwc(merged_normal),     # (H,W,3) in [-1,1]
        'roughness': to_np_hwc(merged_roughness),  # (H,W,1) in [0,1]
        'metallic' : to_np_hwc(merged_metallic),   # (H,W,1) in [0,1]
    }


print('MatForge tile-and-merge inference function ready.')

MatForge tile-and-merge inference function ready.


### 4.3 MatForge — evaluation loop

In [9]:
# MatForge receives the full 1024×1024 image via tile-and-merge.
# Metrics are computed on the full resolution, then centre-cropped
# to 256×256 for the restricted comparison table against Pix2Pix / DeepPBR.

rows_mf = []

for idx, row in val_df.iterrows():
    stem = row['filename']

    rgb_np     = load_image_np(RGB_DIR    / stem)   # [0,1]
    normal_gt  = load_normal_np(NORMAL_DIR / stem)  # [-1,1]
    rough_gt   = load_scalar_np(ROUGH_DIR  / stem)  # [0,1]
    metal_path = METAL_DIR / stem
    if metal_path.exists():
        metal_gt = load_scalar_np(metal_path)
    else:
        metal_gt = np.zeros((rough_gt.shape[0], rough_gt.shape[1], 1), dtype=np.float32)

    pred = matforge_infer(rgb_np)

    rows_mf.append({
        'filename'    : stem,
        'group'       : row['functional_group'],
        'normal_mae'  : angular_mae(pred['normal'],    normal_gt),
        'rough_mae'   : scalar_mae(pred['roughness'],  rough_gt),
        'rough_rmse'  : scalar_rmse(pred['roughness'], rough_gt),
        'metal_mae'   : scalar_mae(pred['metallic'],   metal_gt),
        'metal_rmse'  : scalar_rmse(pred['metallic'],  metal_gt),
        'lpips_render': render_lpips(
            rgb_np,
            pred['normal'], pred['roughness'], pred['metallic'],
            normal_gt, rough_gt, metal_gt
        ),
    })

    if idx % 50 == 0:
        print(f'  [{idx+1}/{len(val_df)}] processed')

df_mf = pd.DataFrame(rows_mf)
df_mf.to_csv(OUT_DIR / 'results_matforge.csv', index=False)
print('MatForge evaluation complete.')

  [1/483] processed
  [51/483] processed
  [101/483] processed
  [151/483] processed
  [201/483] processed
  [251/483] processed
  [301/483] processed
  [351/483] processed
  [401/483] processed
  [451/483] processed
MatForge evaluation complete.


---
## 5. SR evaluation — MatForge_SR vs Real-ESRGAN base vs bicubic

### 5.1 RRDBNet definition and checkpoint loading

In [10]:
class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat: int = 64, num_grow_ch: int = 32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat,                 num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat +   num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2*num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3*num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4*num_grow_ch, num_feat,    3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
        for m in [self.conv1, self.conv2, self.conv3, self.conv4, self.conv5]:
            nn.init.kaiming_normal_(m.weight, a=0.2)
            m.weight.data *= 0.1

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x


class RRDB(nn.Module):
    def __init__(self, num_feat: int = 64, num_grow_ch: int = 32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.rdb3(self.rdb2(self.rdb1(x)))
        return out * 0.2 + x


class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64,
                 num_block=6, num_grow_ch=32, scale=4):
        super().__init__()
        self.scale      = scale
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body       = nn.Sequential(*[RRDB(num_feat, num_grow_ch)
                                          for _ in range(num_block)])
        self.conv_body  = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1   = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2   = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr    = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last  = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu      = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat      = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat      = feat + body_feat
        feat      = self.lrelu(self.conv_up1(
                        F.interpolate(feat, scale_factor=2, mode='nearest')))
        feat      = self.lrelu(self.conv_up2(
                        F.interpolate(feat, scale_factor=2, mode='nearest')))
        return self.conv_last(self.lrelu(self.conv_hr(feat)))


def build_sr_model(ckpt_path: Path, num_block: int) -> RRDBNet:
    net  = RRDBNet(num_block=num_block).to(DEVICE).to(DTYPE)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    if 'params_ema' in ckpt:
        state = ckpt['params_ema']
    elif 'params' in ckpt:
        state = ckpt['params']
    else:
        state = ckpt
    missing, unexpected = net.load_state_dict(state, strict=False)
    params = sum(p.numel() for p in net.parameters()) / 1e6
    print(f'Loaded  blocks={num_block}  params={params:.2f}M  '
          f'missing={len(missing)}  unexpected={len(unexpected)}')
    net.eval()
    return net


# Fine-tuned: 6 blocks (Real-ESRGAN+, trained in matforge-sr-01-training.ipynb)
sr_ft   = build_sr_model(SR_FT_CKPT,   num_block=6)
# Base: 23 blocks (official Real-ESRGAN release, no fine-tuning)
sr_base = build_sr_model(SR_BASE_CKPT, num_block=23)

print('SR models ready.')

Loaded  blocks=6  params=4.47M  missing=0  unexpected=510
Loaded  blocks=23  params=16.70M  missing=0  unexpected=0
SR models ready.


### 5.2 SR inference with tile-and-merge

In [11]:
SR_TILE   = 256
SR_STRIDE = 128
SR_PAD    = SR_TILE // 2
_SR_HANN  = _hann2d(SR_TILE).unsqueeze(0).to(DEVICE)   # (1,256,256)


@torch.no_grad()
def sr_infer(model: RRDBNet, lr_np: np.ndarray) -> np.ndarray:
    """Tile-and-merge SR inference matching the sr.py pipeline.

    Input:  float32 numpy (H, W, 3) in [0, 1] — low-resolution image
    Output: float32 numpy (H*4, W*4, 3) in [0, 1] — super-resolved image
    """
    h, w = lr_np.shape[:2]
    t    = torch.from_numpy(lr_np).permute(2, 0, 1).unsqueeze(0).to(DEVICE).to(DTYPE)
    t_pad = F.pad(t, (SR_PAD, SR_PAD, SR_PAD, SR_PAD), mode='reflect')
    _, _, hp, wp = t_pad.shape

    scale = 4
    hp_sr, wp_sr = hp * scale, wp * scale
    tile_sr = SR_TILE * scale
    stride_sr = SR_STRIDE * scale
    pad_sr    = SR_PAD * scale

    acc_rgb    = torch.zeros(3, hp_sr, wp_sr, device=DEVICE, dtype=DTYPE)
    acc_weight = torch.zeros(1, hp_sr, wp_sr, device=DEVICE, dtype=DTYPE)
    hann_sr    = _hann2d(tile_sr).unsqueeze(0).to(DEVICE)

    ys = list(range(0, hp - SR_TILE + 1, SR_STRIDE))
    xs = list(range(0, wp - SR_TILE + 1, SR_STRIDE))
    if ys[-1] + SR_TILE < hp: ys.append(hp - SR_TILE)
    if xs[-1] + SR_TILE < wp: xs.append(wp - SR_TILE)

    for y in ys:
        for x in xs:
            tile = t_pad[:, :, y:y+SR_TILE, x:x+SR_TILE]
            sr   = model(tile).clamp(0.0, 1.0)
            ys_sr, xs_sr = y * scale, x * scale
            acc_rgb[   :, ys_sr:ys_sr+tile_sr, xs_sr:xs_sr+tile_sr] += sr[0] * hann_sr
            acc_weight[:, ys_sr:ys_sr+tile_sr, xs_sr:xs_sr+tile_sr] += hann_sr

    merged = (acc_rgb / (acc_weight + 1e-8)).clamp(0.0, 1.0)
    # Remove padding (in SR space)
    merged = merged[:, pad_sr:pad_sr + h*scale, pad_sr:pad_sr + w*scale]
    return merged.cpu().float().numpy().transpose(1, 2, 0)


def bicubic_upscale(lr_np: np.ndarray, scale: int = 4) -> np.ndarray:
    """Bicubic upscaling baseline using PIL Lanczos resampling."""
    h, w = lr_np.shape[:2]
    img  = Image.fromarray((lr_np * 255.0).clip(0, 255).astype(np.uint8))
    img  = img.resize((w * scale, h * scale), Image.LANCZOS)
    return np.array(img).astype(np.float32) / 255.0


print('SR inference functions ready.')

SR inference functions ready.


### 5.3 SR evaluation loop

In [12]:
# For each texture in the SR subset:
# 1. HR ground truth = original 1024×1024 image
# 2. LR input = HR downscaled ×4 with bicubic degradation
# 3. Three upscaling conditions are evaluated against HR GT

rows_sr = []

for idx, row in sr_df.iterrows():
    stem   = row['filename']
    hr_np  = load_image_np(RGB_DIR / stem)   # (1024,1024,3) in [0,1]

    # Deterministic bicubic degradation ×4
    h, w   = hr_np.shape[:2]
    lr_pil = Image.fromarray((hr_np * 255.0).astype(np.uint8)).resize(
        (w // 4, h // 4), Image.BICUBIC
    )
    lr_np  = np.array(lr_pil).astype(np.float32) / 255.0

    # --- Three upscaling conditions ---
    bicubic_np = bicubic_upscale(lr_np, scale=4)
    sr_base_np = sr_infer(sr_base, lr_np)
    sr_ft_np   = sr_infer(sr_ft,   lr_np)

    def sr_metrics(pred: np.ndarray, gt: np.ndarray) -> dict:
        """Compute PSNR, SSIM and LPIPS for a pair of RGB images in [0, 1]."""
        psnr = psnr_fn(gt, pred, data_range=1.0)
        ssim = ssim_fn(gt, pred, data_range=1.0, channel_axis=-1)
        lp   = compute_lpips(pred, gt)
        return {'psnr': psnr, 'ssim': ssim, 'lpips': lp}

    m_bic  = sr_metrics(bicubic_np, hr_np)
    m_base = sr_metrics(sr_base_np, hr_np)
    m_ft   = sr_metrics(sr_ft_np,   hr_np)

    rows_sr.append({
        'filename'       : stem,
        'group'          : row['functional_group'],
        'psnr_bicubic'   : m_bic['psnr'],
        'ssim_bicubic'   : m_bic['ssim'],
        'lpips_bicubic'  : m_bic['lpips'],
        'psnr_realesrgan': m_base['psnr'],
        'ssim_realesrgan': m_base['ssim'],
        'lpips_realesrgan': m_base['lpips'],
        'psnr_sr_ft'     : m_ft['psnr'],
        'ssim_sr_ft'     : m_ft['ssim'],
        'lpips_sr_ft'    : m_ft['lpips'],
    })

    if idx % 20 == 0:
        print(f'  [{idx+1}/{len(sr_df)}] processed')

df_sr = pd.DataFrame(rows_sr)
df_sr.to_csv(OUT_DIR / 'results_sr.csv', index=False)
print('SR evaluation complete.')

  [1/100] processed
  [21/100] processed
  [41/100] processed
  [61/100] processed
  [81/100] processed
SR evaluation complete.


---
## 6. Summary tables

In [13]:
# Table 1 — Restricted comparison: Normal + Roughness (all three PBR models)
# Uses only metrics available for Pix2Pix and DeepPBR.

def global_mean(df, cols):
    return df[cols].mean()


pbr_cols = ['normal_mae', 'rough_mae', 'rough_rmse', 'lpips_render']

table1 = pd.DataFrame({
    'Model'         : ['Pix2Pix', 'DeepPBR', 'MatForge'],
    'Normal MAE (°)': [
        df_p2p['normal_mae'].mean(),
        df_dpbr['normal_mae'].mean(),
        df_mf['normal_mae'].mean(),
    ],
    'Roughness MAE' : [
        df_p2p['rough_mae'].mean(),
        df_dpbr['rough_mae'].mean(),
        df_mf['rough_mae'].mean(),
    ],
    'Roughness RMSE': [
        df_p2p['rough_rmse'].mean(),
        df_dpbr['rough_rmse'].mean(),
        df_mf['rough_rmse'].mean(),
    ],
    'LPIPS render'  : [
        df_p2p['lpips_render'].mean(),
        df_dpbr['lpips_render'].mean(),
        df_mf['lpips_render'].mean(),
    ],
})

print('=== Table 1 — Restricted PBR comparison ===')
print(table1.to_string(index=False, float_format='{:.4f}'.format))
table1.to_csv(OUT_DIR / 'table1_pbr_restricted.csv', index=False)

# Table 2 — MatForge full evaluation by group
mf_full_cols = ['normal_mae', 'rough_mae', 'rough_rmse', 'metal_mae', 'metal_rmse', 'lpips_render']
table2 = df_mf.groupby('group')[mf_full_cols].mean().reset_index()
global_row = df_mf[mf_full_cols].mean().to_frame().T
global_row.insert(0, 'group', 'GLOBAL')
table2 = pd.concat([global_row, table2], ignore_index=True)

print('\n=== Table 2 — MatForge full evaluation by material group ===')
print(table2.to_string(index=False, float_format='{:.4f}'.format))
table2.to_csv(OUT_DIR / 'table2_matforge_by_group.csv', index=False)

# Table 3 — SR comparison
sr_summary_cols = [
    'psnr_bicubic', 'ssim_bicubic', 'lpips_bicubic',
    'psnr_realesrgan', 'ssim_realesrgan', 'lpips_realesrgan',
    'psnr_sr_ft', 'ssim_sr_ft', 'lpips_sr_ft',
]
sr_means = df_sr[sr_summary_cols].mean()

table3 = pd.DataFrame({
    'Condition': ['Bicubic', 'Real-ESRGAN base', 'MatForge SR (fine-tuned)'],
    'PSNR ↑'   : [sr_means['psnr_bicubic'],    sr_means['psnr_realesrgan'],    sr_means['psnr_sr_ft']],
    'SSIM ↑'   : [sr_means['ssim_bicubic'],    sr_means['ssim_realesrgan'],    sr_means['ssim_sr_ft']],
    'LPIPS ↓'  : [sr_means['lpips_bicubic'],   sr_means['lpips_realesrgan'],   sr_means['lpips_sr_ft']],
})

print('\n=== Table 3 — SR comparison ===')
print(table3.to_string(index=False, float_format='{:.4f}'.format))
table3.to_csv(OUT_DIR / 'table3_sr.csv', index=False)

=== Table 1 — Restricted PBR comparison ===
   Model  Normal MAE (°)  Roughness MAE  Roughness RMSE  LPIPS render
 Pix2Pix         14.7405         0.2120          0.2327        0.2988
 DeepPBR         13.6414         0.2238          0.2472        0.3392
MatForge         10.3950         0.1084          0.1268        0.2911

=== Table 2 — MatForge full evaluation by material group ===
           group  normal_mae  rough_mae  rough_rmse  metal_mae  metal_rmse  lpips_render
          GLOBAL     10.3950     0.1084      0.1268     0.0410      0.0485        0.2911
brick_terracotta     11.5615     0.0816      0.0993     0.0120      0.0157        0.2732
  ceramic_ground      8.9575     0.0990      0.1188     0.0226      0.0271        0.2373
concrete_plaster     13.2098     0.0992      0.1196     0.1347      0.1596        0.3961
   marble_smooth      7.5648     0.1285      0.1455     0.0705      0.0739        0.2942
           metal      8.2282     0.0943      0.1103     0.1092      0.1527      

---
## 7. Visual grids

In [14]:
# Reloads TF models exclusively to generate predictions for the visual grid.
# Results are saved to disk and TF is released immediately after.

import tensorflow as tf
import gc

# --- Reinstantiate and restore Pix2Pix ---
pix2pix_gen_vis  = Generator()
pix2pix_disc_vis = Discriminator()
gen_opt_vis      = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
disc_opt_vis     = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

ckpt_vis_p2p = tf.train.Checkpoint(
    generator_optimizer=gen_opt_vis,
    discriminator_optimizer=disc_opt_vis,
    generador=pix2pix_gen_vis,
    discriminador=pix2pix_disc_vis,
)
ckpt_vis_p2p.restore(
    str(PIX2PIX_CKPT_DIR / PIX2PIX_CKPT_NAME)
).expect_partial()

# --- Reinstantiate and restore DeepPBR ---
deeppbr_gen_vis  = build_deep_pbr_generator()
deeppbr_disc_vis = build_discriminator()
gen_opt_vis2     = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
disc_opt_vis2    = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

ckpt_vis_dpbr = tf.train.Checkpoint(
    generator_optimizer=gen_opt_vis2,
    discriminator_optimizer=disc_opt_vis2,
    generador_pbr=deeppbr_gen_vis,
    discriminador_pbr=deeppbr_disc_vis,
)
ckpt_vis_dpbr.restore(
    str(DEEPPBR_CKPT_DIR / DEEPPBR_CKPT_NAME)
).expect_partial()

print('TF models reloaded for visual grid.')

# --- Generate and save predictions for the 6 selected stems ---
vis_preds_dir = OUT_DIR / 'vis_preds'
vis_preds_dir.mkdir(exist_ok=True)

# selected_stems is defined in cell 7 — compute it here first
# using the same logic so both cells are consistent
sr_sorted_vis    = df_mf.sort_values('lpips_render').reset_index(drop=True)
metal_vis        = df_mf[df_mf['group'] == 'metal'].nsmallest(1, 'lpips_render')
non_metal_vis    = df_mf[df_mf['group'] != 'metal'].copy()
non_metal_sorted = non_metal_vis.sort_values('lpips_render').reset_index(drop=True)
indices_vis      = np.linspace(0, len(non_metal_sorted) - 1, 5, dtype=int)
selected_stems   = pd.concat(
    [metal_vis, non_metal_sorted.iloc[indices_vis]]
).reset_index(drop=True)['filename'].tolist()

print(f'Selected stems: {selected_stems}')

for stem in selected_stems:
    rgb_np   = load_image_np(RGB_DIR / stem)
    rgb_crop = centre_crop_256(rgb_np)
    tf_input = rgb_crop * 2.0 - 1.0
    tf_t     = tf.expand_dims(tf.constant(tf_input, dtype=tf.float32), 0)

    # Pix2Pix
    p2p_out   = pix2pix_gen_vis(tf_t, training=False)[0].numpy()
    p2p_norm  = p2p_out[:, :, :3]                       # [-1,1]
    p2p_rough = (p2p_out[:, :, 3:4] + 1.0) / 2.0       # [0,1]
    np.save(vis_preds_dir / f'{stem}_p2p_normal.npy',   p2p_norm)
    np.save(vis_preds_dir / f'{stem}_p2p_rough.npy',    p2p_rough)

    # DeepPBR
    dpbr_norm_t, dpbr_rough_t = deeppbr_gen_vis(tf_t, training=False)
    dpbr_norm  = dpbr_norm_t[0].numpy()                 # [-1,1]
    dpbr_rough = (dpbr_rough_t[0].numpy() + 1.0) / 2.0 # [0,1]
    np.save(vis_preds_dir / f'{stem}_dpbr_normal.npy',  dpbr_norm)
    np.save(vis_preds_dir / f'{stem}_dpbr_rough.npy',   dpbr_rough)

print('Predictions saved. Releasing TF memory.')

del pix2pix_gen_vis, pix2pix_disc_vis, deeppbr_gen_vis, deeppbr_disc_vis
del ckpt_vis_p2p, ckpt_vis_dpbr
del gen_opt_vis, gen_opt_vis2, disc_opt_vis, disc_opt_vis2
gc.collect()
tf.keras.backend.clear_session()
print('TF released.')

/tmp/ipykernel_23/688787722.py:54: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


TF models reloaded for visual grid.
Selected stems: ['metal_0175.png', 'stone_0201.png', 'ceramic_0494.png', 'stone_0480.png', 'terracotta_0166.png', 'concrete_0180.png']
Predictions saved. Releasing TF memory.
TF released.


In [15]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec


def norm_to_01(n):
    return ((n + 1.0) / 2.0).clip(0, 1)


# Compute selected stems using MatForge LPIPS as ranking criterion
metal_sel     = df_mf[df_mf['functional_group'] == 'metal'].nsmallest(1, 'lpips_render')
non_metal_sel = df_mf[df_mf['functional_group'] != 'metal'].copy()
non_metal_sorted = non_metal_sel.sort_values('lpips_render').reset_index(drop=True)
indices_sel   = np.linspace(0, len(non_metal_sorted) - 1, 5, dtype=int)
selected_stems = pd.concat(
    [metal_sel, non_metal_sorted.iloc[indices_sel]]
).reset_index(drop=True)['filename'].tolist()

print(f'Selected textures: {selected_stems}')

row_labels = ['GT', 'Pix2Pix', 'DeepPBR', 'MatForge']
col_labels = ['Color', 'Normal', 'Roughness']

for stem in selected_stems:
    # --- Load GT ---
    rgb_np    = load_image_np(RGB_DIR    / stem)
    normal_gt = load_normal_np(NORMAL_DIR / stem)
    rough_gt  = load_scalar_np(ROUGH_DIR  / stem)

    rgb_crop    = centre_crop_256(rgb_np)
    normal_crop = centre_crop_256(normal_gt)
    rough_crop  = centre_crop_256(rough_gt)

    # --- TF inference ---
    tf_t = tf.expand_dims(
        tf.constant(rgb_crop * 2.0 - 1.0, dtype=tf.float32), 0
    )

    p2p_out   = pix2pix_gen(tf_t, training=False)[0].numpy()
    p2p_norm  = p2p_out[:, :, :3]               # [-1,1]
    p2p_rough = (p2p_out[:, :, 3:4] + 1.0) / 2.0  # [0,1]

    dpbr_norm_t, dpbr_rough_t = deeppbr_gen(tf_t, training=False)
    dpbr_norm  = dpbr_norm_t[0].numpy()             # [-1,1]
    dpbr_rough = (dpbr_rough_t[0].numpy() + 1.0) / 2.0  # [0,1]

    # --- MatForge inference ---
    mf_pred = matforge_infer(rgb_np)
    mf_norm  = centre_crop_256(mf_pred['normal'])    # [-1,1]
    mf_rough = centre_crop_256(mf_pred['roughness']) # [0,1]

    # --- Build grid: 4 rows x 3 cols ---
    data = [
        # GT
        [rgb_crop,
         norm_to_01(normal_crop),
         np.repeat(rough_crop, 3, axis=-1)],
        # Pix2Pix
        [rgb_crop,
         norm_to_01(p2p_norm),
         np.repeat(p2p_rough, 3, axis=-1)],
        # DeepPBR
        [rgb_crop,
         norm_to_01(dpbr_norm),
         np.repeat(dpbr_rough, 3, axis=-1)],
        # MatForge
        [rgb_crop,
         norm_to_01(mf_norm),
         np.repeat(mf_rough, 3, axis=-1)],
    ]

    fig, axes = plt.subplots(4, 3, figsize=(10, 13))

    for row_idx, (row_data, row_label) in enumerate(zip(data, row_labels)):
        for col_idx, (img, col_label) in enumerate(zip(row_data, col_labels)):
            ax = axes[row_idx, col_idx]
            ax.imshow(img.clip(0, 1))
            ax.axis('off')
            if row_idx == 0:
                ax.set_title(col_label, fontsize=11, fontweight='bold', pad=6)
            if col_idx == 0:
                ax.set_ylabel(row_label, fontsize=11, fontweight='bold',
                              rotation=90, labelpad=8, va='center')

    fig.suptitle(stem.replace('.png', ''), fontsize=12, y=1.01)
    plt.tight_layout()

    save_path = OUT_DIR / f'grid_{stem.replace(".png", "")}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

print('All visual grids saved.')

KeyError: 'functional_group'

---
## 8. SR visual grid

In [ ]:
# SR visual comparison: 4 textures at evenly spaced LPIPS percentiles
# Layout: 1 figure per texture, 3 rows x 3 cols
# Rows: Bicubic | Real-ESRGAN base | MatForge SR
# Cols: Full image (256 crop) | Zoomed patch | Delta vs HR GT

sr_row_labels = ['Bicubic', 'Real-ESRGAN base', 'MatForge SR']
sr_col_labels = ['Upscaled (crop)', 'Detail (zoom)', 'HR GT (crop)']

sr_sorted  = df_sr.sort_values('lpips_sr_ft').reset_index(drop=True)
sr_indices = np.linspace(0, len(sr_sorted) - 1, 4, dtype=int)
sr_stems   = sr_sorted.iloc[sr_indices]['filename'].tolist()

print(f'SR selected textures: {sr_stems}')

ZOOM_SIZE = 64   # pixel size of the zoomed patch

for stem in sr_stems:
    hr_np  = load_image_np(RGB_DIR / stem)
    lr_pil = Image.fromarray((hr_np * 255.0).astype(np.uint8)).resize(
        (hr_np.shape[1] // 4, hr_np.shape[0] // 4), Image.BICUBIC
    )
    lr_np = np.array(lr_pil).astype(np.float32) / 255.0

    bic_np  = bicubic_upscale(lr_np, scale=4)
    base_np = sr_infer(sr_base, lr_np)
    ft_np   = sr_infer(sr_ft,   lr_np)

    hr_crop  = centre_crop_256(hr_np)
    bic_crop = centre_crop_256(bic_np)
    base_crop = centre_crop_256(base_np)
    ft_crop  = centre_crop_256(ft_np)

    # Zoom patch: centre 64×64 of the crop
    def zoom(img):
        h, w = img.shape[:2]
        y0   = (h - ZOOM_SIZE) // 2
        x0   = (w - ZOOM_SIZE) // 2
        patch = img[y0:y0+ZOOM_SIZE, x0:x0+ZOOM_SIZE]
        # Upscale patch for visibility
        return np.array(
            Image.fromarray((patch * 255).clip(0, 255).astype(np.uint8))
            .resize((256, 256), Image.NEAREST)
        ).astype(np.float32) / 255.0

    data_sr = [
        [bic_crop,  zoom(bic_crop),  hr_crop],
        [base_crop, zoom(base_crop), hr_crop],
        [ft_crop,   zoom(ft_crop),   hr_crop],
    ]

    fig, axes = plt.subplots(3, 3, figsize=(10, 10))

    for row_idx, (row_data, row_label) in enumerate(zip(data_sr, sr_row_labels)):
        for col_idx, (img, col_label) in enumerate(zip(row_data, sr_col_labels)):
            ax = axes[row_idx, col_idx]
            ax.imshow(img.clip(0, 1))
            ax.axis('off')
            if row_idx == 0:
                ax.set_title(col_label, fontsize=11, fontweight='bold', pad=6)
            if col_idx == 0:
                ax.set_ylabel(row_label, fontsize=11, fontweight='bold',
                              rotation=90, labelpad=8, va='center')

    fig.suptitle(stem.replace('.png', ''), fontsize=12, y=1.01)
    plt.tight_layout()

    save_path = OUT_DIR / f'sr_grid_{stem.replace(".png", "")}.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')

print('All SR grids saved.')

---
## 9. Output summary

In [ ]:
print('=== Benchmark complete — output files ===')
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {size_kb:>8.1f} KB')